ECEN 743: Assignment 6
==============

**Author:** *Brody Jordan*\
**UIN:** 431005357\
**Description:** In this assignment, you will implement and evaluate modern Deep Reinforcement Learning algorithms on high-dimensional continuous control tasks. You will use the MuJoCo environments from Gymnasium, specifically focusing on the Pusher and Humanoid environments.

---

# Report 

# Code

In [1]:
import os
os.environ["MUJOCO_GL"] = "osmesa"

import sys
print(sys.executable)

C:\Users\bljor\anaconda3\envs\rl\python.exe


## 1. Environment Setup

In [2]:
import os
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from gymnasium.wrappers import RecordVideo

In [3]:
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env

In [4]:
ALGO_MAP = {"ppo": PPO, "sac": SAC}
N_CPUS = os.cpu_count() or 4


class _RewardLogger(BaseCallback):
    """Collects per-episode rewards so we can plot them later."""

    def __init__(self):
        super().__init__()
        self.rewards, self.timesteps = [], []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.rewards.append(info["episode"]["r"])
                self.timesteps.append(self.num_timesteps)
        return True


def train(env_name: str, algo_name: str, total_timesteps: int = 1_000_000,
          seed: int = 0, n_envs: int | None = None, **kwargs) -> dict:
    """Train an RL agent with max hardware utilization."""
    algo_key = algo_name.lower()
    is_ppo = algo_key == "ppo"

    # --- parallel envs across CPU cores ---
    if n_envs is None:
        n_envs = min(N_CPUS, 16) if is_ppo else min(N_CPUS, 8)
    vec_env = make_vec_env(env_name, n_envs=n_envs, seed=seed,
                           vec_env_cls=SubprocVecEnv)

    # PPO MlpPolicy is faster on CPU; SAC benefits from GPU batch updates
    device = "cpu" if is_ppo else ("cuda" if torch.cuda.is_available() else "cpu")

    # avoid thread oversubscription when many env subprocesses share CPU
    if device == "cpu":
        torch.set_num_threads(max(1, N_CPUS // n_envs))

    # throughput-oriented defaults (anything in kwargs overrides these)
    if is_ppo:
        kwargs.setdefault("n_steps", 2048)
        kwargs.setdefault("batch_size", n_envs * 64)   # scales with cores
        kwargs.setdefault("n_epochs", 10)
    else:
        kwargs.setdefault("batch_size", 1024)           # fill GPU SMs
        kwargs.setdefault("gradient_steps", -1)         # 1 grad step per env step

    print(f"\n{'='*60}")
    print(f"  {algo_key.upper()} on {env_name}")
    print(f"  device={device}  n_envs={n_envs}  timesteps={total_timesteps:,}")
    print(f"  {kwargs}")
    print(f"{'='*60}\n")

    model = ALGO_MAP[algo_key](
        "MlpPolicy", vec_env, seed=seed, device=device, verbose=1, **kwargs
    )

    logger = _RewardLogger()
    model.learn(total_timesteps=total_timesteps, callback=logger, progress_bar=True)
    vec_env.close()

    path = f"models/{algo_key}_{env_name}"
    os.makedirs("models", exist_ok=True)
    model.save(path)
    print(f"Model saved → {path}")

    return dict(
        model=model,
        rewards=logger.rewards,
        timesteps=logger.timesteps,
        env_name=env_name,
        algo_name=algo_name.upper(),
    )


In [23]:
def report(result: dict, video_folder: str = "videos", smooth_window: int = 50):
    """Plot the training curve and save a video of the trained agent."""
    rewards = np.array(result["rewards"])
    timesteps = np.array(result["timesteps"])
    label = f"{result['algo_name']} on {result['env_name']}"

    # ---- training curve ----
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(timesteps, rewards, alpha=0.25, color="steelblue", label="Episode reward")
    if len(rewards) >= smooth_window:
        k = np.ones(smooth_window) / smooth_window
        smoothed = np.convolve(rewards, k, mode="valid")
        ax.plot(
            timesteps[smooth_window - 1 :],
            smoothed,
            color="orangered",
            lw=2,
            label=f"Smoothed (w={smooth_window})",
        )
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Episode Reward")
    ax.set_title(label)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

    # ---- record one episode as video ----
    os.makedirs(video_folder, exist_ok=True)
    prefix = f"{result['algo_name']}_{result['env_name']}"
    env = gym.make(result["env_name"], render_mode="rgb_array")
    env = RecordVideo(
        env, video_folder=video_folder, name_prefix=prefix,
        episode_trigger=lambda _: True,
    )

    model = result["model"]
    obs, _ = env.reset()
    done, ep_reward = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, _ = env.step(action)
        ep_reward += r
        done = terminated or truncated
    env.close()
    print(f"Video saved → {video_folder}/{prefix}-episode-0.mp4  |  Reward: {ep_reward:.1f}")
        

## 2. Algorithm Training & Hyperparameter Tuning

### 2(a). PPO

#### Pusher

In [20]:
ppo_pusher = train("Pusher-v5", "ppo", total_timesteps=1_000_000)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [9]:
report(ppo_pusher)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


C:\Users\bljor\anaconda3\envs\rl\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22       |
|    ep_rew_mean     | 22       |
| time/              |          |
|    fps             | 45       |
|    iterations      | 1        |
|    time_elapsed    | 44       |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 26.1        |
|    ep_rew_mean          | 26.1        |
| time/                   |             |
|    fps                  | 45          |
|    iterations           | 2           |
|    time_elapsed         | 89          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008181183 |
|    clip_fraction        | 0.0717      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.687      |
|    explained_variance   | 0.00108     |
|    learning_rate        | 0.

KeyboardInterrupt: 

#### Humanoid

In [ ]:
ppo_humanoid = train("Humanoid-v5", "ppo", total_timesteps=5_000_000)

In [ ]:
report(ppo_humanoid)

### 2(b). SAC

#### Pusher

In [ ]:
sac_pusher = train("Pusher-v5", "sac", total_timesteps=500_000)

In [ ]:
report(sac_pusher)

#### Humanoid

In [ ]:
sac_humanoid = train("Humanoid-v5", "sac", total_timesteps=3_000_000)

In [ ]:
report(sac_humanoid)

## 3. Inference & Video Generation